In [16]:
import pandas as pd

In [17]:
market = pd.read_parquet('../data/raw/news/GOOGL.parquet')

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/news/GOOGL.parquet'

In [ ]:
market

,ticker,published_at,source,category,headline,summary,url
0,GOOGL,2025-12-31 22:13:37+00:00,CNBC,company,"Dust to data centers: The year AI tech giants,...",Big Tech is remaking the U.S. map into an AI e...,https://finnhub.io/api/news?id=cb398d7ed860723...
1,GOOGL,2025-12-31 17:27:07+00:00,SeekingAlpha,company,Tesla: Challenging 2026 Ahead,"Tesla, Inc. rated Strong Sell: weakening deman...",https://finnhub.io/api/news?id=6fca9458cab80a8...
2,GOOGL,2025-12-31 16:42:05+00:00,Yahoo,company,Alphabet Inc. (GOOGL) Acquires Intersection of...,Alphabet Inc. (NASDAQ:GOOGL) is among the Best...,https://finnhub.io/api/news?id=846c1a0b9802dc5...
3,GOOGL,2025-12-31 16:33:41+00:00,Yahoo,company,PTRN: Jefferies Warns of Margin Pressure and A...,Pattern Group Inc. (NASDAQ:PTRN) is one of the...,https://finnhub.io/api/news?id=37dc1b2df7fbf2c...
4,GOOGL,2025-12-31 16:15:24+00:00,SeekingAlpha,company,"Atlassian: Strong Fundamentals, Temporary Mark...",Atlassian Corporation's Q1 2026 shows robust g...,https://finnhub.io/api/news?id=272cbd2d3ce3b1b...
...,...,...,...,...,...,...,...
244,GOOGL,2025-12-22 20:58:25+00:00,Yahoo,company,Sector Update: Health Care Stocks Higher Late ...,Health care stocks advanced late Monday aftern...,https://finnhub.io/api/news?id=08aa068089fdd0f...
245,GOOGL,2025-12-22 20:53:13+00:00,Yahoo,company,Sector Update: Tech Stocks Climb Late Afternoon,"Tech stocks were higher late Monday afternoon,...",https://finnhub.io/api/news?id=50ccf80ebea224a...
246,GOOGL,2025-12-22 20:35:00+00:00,Yahoo,company,Tesla Stock Poised for Even More Record Highs,TSLA tends to outperform in December,https://finnhub.io/api/news?id=f83473eb57a830b...
247,GOOGL,2025-12-22 20:21:39+00:00,SeekingAlpha,company,Tracking Cliff Asness' AQR Capital Management ...,AQR Capital Managementâs Q3 2025 13F portfol...,https://finnhub.io/api/news?id=a4fd157bfb3468b...


In [ ]:
market.iloc[0]['summary']

'2025 was a rollercoaster for Benzinga chat room participants, with mixed opinions on various stocks; 2026 may see escalation of China/Taiwan issue.'

In [ ]:
import os

os.chdir('..')

In [ ]:
from alphaduel import TradingEnv, TradingEnvConfig

In [ ]:
env = TradingEnv(
    config=TradingEnvConfig(
        start="2020-01-01",
        end="2024-12-31",
        tickers=["AAPL", "MSFT", "NVDA"],
        initial_cash=100_000,
        transaction_fee=0.001,   # 10 bps
        n_days=252,
        max_shares=10,
        features=["close", "pct_change_1d", "pct_change_5d", "rsi_14", "volatility_20d"],
        include_news=False,
    ),
)


In [ ]:
env.reset(seed=0)

({'cash': array([100000.], dtype=float32),
  'holdings': array([0., 0., 0.], dtype=float32),
  'portfolio_value': array([100000.], dtype=float32),
  'weights': array([0., 0., 0.], dtype=float32),
  'features': array([[ 7.4539909e+01, -4.4338450e-02, -2.7646033e-02,  4.0781033e+01,
           1.7679043e-02],
         [ 1.6091362e+02, -1.4758855e-02,  3.1447120e-02,  4.9823853e+01,
           1.2970104e-02],
         [ 5.8772964e+00, -3.8159195e-02, -5.6092411e-02,  3.0994764e+01,
           1.8825818e-02]], dtype=float32)},
 {'date': '2020-01-31',
  'tickers': ['AAPL', 'MSFT', 'NVDA'],
  'feature_names': ['close',
   'pct_change_1d',
   'pct_change_5d',
   'rsi_14',
   'volatility_20d'],
  'executed': array([0, 0, 0]),
  'fees_paid': 0.0,
  'cash': 100000.0,
  'holdings': array([0., 0., 0.]),
  'portfolio_value': 100000.0,
  'step': 0})

In [ ]:
env.action_space.sample()

array([-6, -4,  2], dtype=int32)

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=" ",
    temperature=0,
    # other params...
)


from pydantic import BaseModel, Field
from typing import Literal

class ExampleResponse(BaseModel):
    choice: str = Field(description="The choice of the user")
    stock: Literal["AAPL", "MSFT", "NVDA"] = Field(description="The stock that the user chose")


chain = llm.with_structured_output(ExampleResponse)


chain.invoke([
    {
        "role": "system",
        "content": "You are an expert in stock market analysis ....."
    },

    {
        "role": "user",
        "content": "What do you do?"
    }
])

ExampleResponse(choice='I am Qwen3.5, an advanced AI model designed to provide high-quality responses across various domains.', stock='AAPL')

In [ ]:
obs, info = env.reset(seed=0)
obs, reward, terminated, truncated, info = env.step(env.action_space.sample())

In [1]:
import os
os.chdir('..')

In [2]:
from alphaduel import TradingEnv, TradingEnvConfig, LLMPolicy

env = TradingEnv(config=TradingEnvConfig(
    start="2024-01-02", end="2024-03-29",
    tickers=["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN", "META"],
    initial_cash=10000, n_days=20, max_shares=5,
))

policy = LLMPolicy.from_env(env)  # model from configs/project.yaml → llm.*

obs, info = env.reset(seed=0)
done = False


In [3]:
action = policy.act(obs, info)          # int32[n_tickers]

2026-07-27 06:03:10 | INFO     | alphaduel.strategies.llm_policy | LLM decision date=2024-01-31 action=[0, 0, 2, 0, 0, 0] rationale_chars=984


In [4]:
info

{'date': '2024-01-31',
 'tickers': ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META'],
 'feature_names': ['close',
  'pct_change_1d',
  'pct_change_5d',
  'rsi_14',
  'volatility_20d',
  'sma_ratio_20d'],
 'executed': array([0, 0, 0, 0, 0, 0]),
 'fees_paid': 0.0,
 'cash': 10000.0,
 'holdings': array([0., 0., 0., 0., 0., 0.]),
 'portfolio_value': 10000.0,
 'step': 0}

In [5]:
obs

{'cash': array([10000.], dtype=float32),
 'holdings': array([0., 0., 0., 0., 0., 0.], dtype=float32),
 'portfolio_value': array([10000.], dtype=float32),
 'weights': array([0., 0., 0., 0., 0., 0.], dtype=float32),
 'features': array([[ 1.8233606e+02, -1.9357665e-02, -5.1928021e-02,  3.3682724e+01,
          1.3366322e-02,  9.8174661e-01],
        [ 3.9000244e+02, -2.6946284e-02, -1.2370867e-02,  5.9481220e+01,
          1.0424189e-02,  1.0184462e+00],
        [ 6.1417835e+01, -1.9864758e-02,  2.6890221e-03,  6.6458191e+01,
          1.9785687e-02,  1.0858265e+00],
        [ 1.3886862e+02, -7.5003259e-02, -5.7834480e-02,  4.6692146e+01,
          2.1048995e-02,  9.7206610e-01],
        [ 1.5520000e+02, -2.3899389e-02, -1.0645746e-02,  4.2974243e+01,
          1.3854731e-02,  1.0091060e+00],
        [ 3.8674377e+02, -2.4796214e-02, -1.4333767e-03,  6.0667225e+01,
          1.4287770e-02,  1.0395569e+00]], dtype=float32)}

In [6]:
print(policy.build_messages(obs, info)[-1]['content'])

Decide today's trades. Reason in free-form text, then end with a fenced JSON action block:

```json
{"actions": {"TICKER": <int>, "...": <int>}}
```

Date: 2024-01-31
Step: 0
Portfolio value: 10000.0
Cash: 10000.0

Holdings (shares):
- AAPL: 0.0
- MSFT: 0.0
- NVDA: 0.0
- GOOGL: 0.0
- AMZN: 0.0
- META: 0.0

Features by ticker:
- AAPL: {'close': 182.3361, 'pct_change_1d': -0.0194, 'pct_change_5d': -0.0519, 'rsi_14': 33.6827, 'volatility_20d': 0.0134, 'sma_ratio_20d': 0.9817}
- MSFT: {'close': 390.0024, 'pct_change_1d': -0.0269, 'pct_change_5d': -0.0124, 'rsi_14': 59.4812, 'volatility_20d': 0.0104, 'sma_ratio_20d': 1.0184}
- NVDA: {'close': 61.4178, 'pct_change_1d': -0.0199, 'pct_change_5d': 0.0027, 'rsi_14': 66.4582, 'volatility_20d': 0.0198, 'sma_ratio_20d': 1.0858}
- GOOGL: {'close': 138.8686, 'pct_change_1d': -0.075, 'pct_change_5d': -0.0578, 'rsi_14': 46.6921, 'volatility_20d': 0.021, 'sma_ratio_20d': 0.9721}
- AMZN: {'close': 155.2, 'pct_change_1d': -0.0239, 'pct_change_5d': -0.0106

In [13]:
obs, reward, term, trunc, info = env.step(action)
print(policy.last_decision.rationale)
done = term or trunc

The market has rallied significantly today across the board with all six tickers posting positive gains (1% to 2.6%). The RSI levels are elevated for most names—MSFT at nearly 65 and NVDA well above 70—which indicates they have already absorbed much of this momentum without breaking out into new territory, suggesting a "buy the rumor, sell the news" scenario is likely playing out or simply consolidating.

NVDA has been my primary focus due to its earlier strength (+2% daily), but today's rally suggests it might be overextended given such strong gains across the sector and high RSI readings. Adding more now would increase risk without clear upside, so I will trim some of that exposure before looking for better entry points elsewhere in a potential rebound tomorrow.

I'll sell 1 share from NVDA to free up cash while keeping my position intact but slightly reduced weight on this name. The others are too expensive or overbought at current levels, and there's no compelling reason to add mor

In [21]:
action = policy.act(obs, info)          # int32[n_tickers]
print(policy.last_decision.rationale)

obs, reward, term, trunc, info = env.step(action)
done = term or trunc

2026-07-27 06:08:50 | INFO     | alphaduel.strategies.llm_policy | LLM decision date=2024-02-06 action=[0, 0, 0, 0, 0, 0] rationale_chars=1787
The market has settled into a range-bound choppy session with mixed signals across the sector (-0% to +1%). The RSI levels are still elevated for most names (MSFT ~63, NVDA ~76), indicating that while prices have cooled from yesterday's peak, they haven't yet given back all their gains or broken out into new territory. This suggests a "buy the rumor, sell the news" scenario where momentum is exhausted but not fully reversed.

NVDA has been my primary focus due to its earlier strength (+8% 5-day), but today's data shows it at an extremely high RSI of ~76 and only +0.1% daily gain with a slight pullback (-1%). Adding more now would likely result in further downside or stagnation without clear upside, as the momentum is clearly overextended for immediate entry points.

AMZN has shown resilience today (+5-8%) despite being down slightly on the day, 

In [18]:
print(policy.build_messages(obs, info)[-1]['content'])

Decide today's trades. Reason in free-form text, then end with a fenced JSON action block:

```json
{"actions": {"TICKER": <int>, "...": <int>}}
```

Date: 2024-02-05
Step: 3
Portfolio value: 10005.87
Cash: 10005.87

Holdings (shares):
- AAPL: 0.0
- MSFT: 0.0
- NVDA: 0.0
- GOOGL: 0.0
- AMZN: 0.0
- META: 0.0

Features by ticker:
- AAPL: {'close': 185.5794, 'pct_change_1d': 0.0098, 'pct_change_5d': -0.0211, 'rsi_14': 42.6673, 'volatility_20d': 0.0134, 'sma_ratio_20d': 0.9958}
- MSFT: {'close': 397.9186, 'pct_change_1d': -0.0135, 'pct_change_5d': -0.0099, 'rsi_14': 62.8021, 'volatility_20d': 0.0115, 'sma_ratio_20d': 1.0241}
- NVDA: {'close': 69.209, 'pct_change_1d': 0.0479, 'pct_change_5d': 0.1099, 'rsi_14': 81.1363, 'volatility_20d': 0.0217, 'sma_ratio_20d': 1.1681}
- GOOGL: {'close': 142.4171, 'pct_change_1d': 0.0091, 'pct_change_5d': -0.064, 'rsi_14': 53.5939, 'volatility_20d': 0.0206, 'sma_ratio_20d': 0.9913}
- AMZN: {'close': 170.31, 'pct_change_1d': -0.0087, 'pct_change_5d': 0.0561,

In [12]:
print(policy.last_decision.rationale)

The market has rallied significantly today across the board with all six tickers posting positive gains (1% to 2.6%). The RSI levels are elevated for most names—MSFT at nearly 65 and NVDA well above 70—which indicates they have already absorbed much of this momentum without breaking out into new territory, suggesting a "buy the rumor, sell the news" scenario is likely playing out or simply consolidating.

NVDA has been my primary focus due to its earlier strength (+2% daily), but today's rally suggests it might be overextended given such strong gains across the sector and high RSI readings. Adding more now would increase risk without clear upside, so I will trim some of that exposure before looking for better entry points elsewhere in a potential rebound tomorrow.

I'll sell 1 share from NVDA to free up cash while keeping my position intact but slightly reduced weight on this name. The others are too expensive or overbought at current levels, and there's no compelling reason to add mor

In [10]:
action

array([0, 0, 2, 0, 0, 0], dtype=int32)